<a href="https://colab.research.google.com/github/Didarulisalmdidar/QLSTM_Multilayer_Network/blob/main/IntraSectorQLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pennylane

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 78.3 MB/s eta 0:00:00


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pennylane as qml
from torch.utils.data import DataLoader, TensorDataset

BASE_DIR  = "/content/drive/MyDrive/MSC Thesis"
DATA_DIR  = os.path.join(BASE_DIR, "Processed")
MODEL_DIR = os.path.join(BASE_DIR, "Models_Dense")
LOSS_DIR  = os.path.join(BASE_DIR, "Losses_Dense")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOSS_DIR,  exist_ok=True)

In [ ]:
# ── ↓↓↓ CHANGE THIS EACH SESSION ↓↓↓ ─────────────────────
SECTOR = "Healthcare"
# Session 1: "Energy"
# Session 2: "IT"
# Session 3: "Financial"
# Session 4: "Healthcare"
# ── ↑↑↑ CHANGE THIS EACH SESSION ↑↑↑ ─────────────────────

SECTOR_TICKERS = {
    "Energy"    : ["APA","COP","CVX","DVN","EOG","FLR","FTI",
                   "HAL","NFG","OKE","OXY","SLB","VLO","WMB","XOM"],
    "IT"        : ["AAPL","MSFT","NVDA","ORCL","CRM","AMD","QCOM",
                   "TXN","INTC","IBM","CSCO","AMAT","ADI","MU","ADBE"],
    "Financial" : ["JPM","BAC","WFC","GS","MS","C","BLK","SCHW",
                   "AXP","USB","PNC","COF","BK","STT","CME"],
    "Healthcare": ["JNJ","UNH","LLY","ABT","TMO","MRK","DHR","ABBV",
                   "PFE","BMY","AMGN","GILD","SYK","MDT","BSX"],
}

N_QUBITS    = 4
N_LAYERS    = 2
N_STOCKS    = 15
HIDDEN_DIM  = 4
WINDOW_SIZE = 5
BATCH_SIZE  = 16
N_EPOCHS    = 10
LR          = 0.01
N_YEARS     = 20

print(f"✅ Config: {SECTOR} — {N_YEARS} years")

✅ Config: Healthcare — 20 years


In [ ]:
# ── Quantum device ─────────────────────────────────────────
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    DIFF_METHOD = "adjoint"
    print("✅ Using lightning.qubit (fast)")
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    DIFF_METHOD = "backprop"
    print("⚠️ Using default.qubit (slower)")


def vqc(inputs, weights):
    for i in range(N_QUBITS):
        qml.RY(inputs[i], wires=i)
    for i in range(N_QUBITS - 1):
        qml.CNOT(wires=[i, i + 1])
    qml.CNOT(wires=[N_QUBITS - 1, 0])
    for l in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(weights[l, i, 0], wires=i)
            qml.RZ(weights[l, i, 1], wires=i)
        for i in range(N_QUBITS - 1):
            qml.CNOT(wires=[i, i + 1])
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]

qnode = qml.QNode(vqc, dev, interface="torch",
                  diff_method=DIFF_METHOD)

print("✅ Quantum circuit ready")

✅ Using lightning.qubit (fast)
✅ Quantum circuit ready


In [ ]:
# ── qLSTM Cell ─────────────────────────────────────────────
class QuantumLSTMCell(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden_dim        = HIDDEN_DIM
        self.input_proj        = nn.Linear(N_STOCKS + HIDDEN_DIM, N_QUBITS)
        self.weights_forget    = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_input     = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_candidate = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.weights_output    = nn.Parameter(torch.randn(N_LAYERS, N_QUBITS, 2) * 0.1)
        self.output_proj       = nn.Linear(HIDDEN_DIM, N_STOCKS)

    def forward_gate(self, projected, weights):
        results = []
        for b in range(projected.shape[0]):
            out = qnode(projected[b], weights)
            out_tensor = out.float() if isinstance(out, torch.Tensor) \
                         else torch.stack(out).float()
            results.append(out_tensor)
        return torch.stack(results)

    def forward(self, x_t, h_t, c_t):
        combined  = torch.cat([x_t, h_t], dim=-1)
        projected = torch.tanh(self.input_proj(combined)) * torch.pi
        f_t = torch.sigmoid(self.forward_gate(projected, self.weights_forget))
        i_t = torch.sigmoid(self.forward_gate(projected, self.weights_input))
        g_t = torch.tanh(self.forward_gate(projected,    self.weights_candidate))
        o_t = torch.sigmoid(self.forward_gate(projected, self.weights_output))
        c_new = f_t * c_t + i_t * g_t
        h_new = o_t * torch.tanh(c_new)
        return h_new, c_new, self.output_proj(h_new)

In [ ]:
# ── qLSTM Autoencoder WITH Dense Bottleneck ────────────────
class QuantumLSTMAutoencoderDense(nn.Module):
    """
    qLSTM Autoencoder with dense bottleneck.
    W extracted from self.dense.weight — Tuhin et al. method.
    """
    def __init__(self):
        super().__init__()
        self.hidden_dim = HIDDEN_DIM
        self.cell  = QuantumLSTMCell()
        # Dense bottleneck — W extracted directly from here
        self.dense = nn.Linear(N_STOCKS, N_STOCKS, bias=False)

    def forward(self, x):
        h_t = torch.zeros(x.shape[0], self.hidden_dim)
        c_t = torch.zeros(x.shape[0], self.hidden_dim)
        recons = []
        for t in range(x.shape[1]):
            h_t, c_t, recon = self.cell(x[:, t, :], h_t, c_t)
            recon = self.dense(recon)    # ← dense bottleneck
            recons.append(recon)
        return torch.stack(recons, dim=1)

print("✅ Model with dense bottleneck defined")

✅ Model with dense bottleneck defined


In [ ]:
# ── Helper functions ───────────────────────────────────────
def load_year(sector, year_idx):
    path = os.path.join(DATA_DIR, sector, f"year_{year_idx:02d}.csv")
    df   = pd.read_csv(path)
    arr  = df.drop(columns=["window_id","day"]).values.reshape(
               -1, WINDOW_SIZE, N_STOCKS)
    return torch.tensor(arr, dtype=torch.float32)


def get_model_path(sector, year_idx):
    return os.path.join(MODEL_DIR, sector,
                        f"year_{year_idx:02d}.pt")


def is_trained(sector, year_idx):
    return os.path.exists(get_model_path(sector, year_idx))


def train_one(sector, year_idx):
    x       = load_year(sector, year_idx)
    loader  = DataLoader(TensorDataset(x),
                         batch_size=BATCH_SIZE, shuffle=True)
    model   = QuantumLSTMAutoencoderDense()
    optim   = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()
    losses  = []

    for epoch in range(N_EPOCHS):
        epoch_loss, n = 0.0, 0
        for (batch,) in loader:
            optim.zero_grad()
            loss = loss_fn(model(batch), batch)
            loss.backward()
            optim.step()
            epoch_loss += loss.item()
            n += 1
        avg = epoch_loss / n
        losses.append(avg)
        print(f"      Epoch {epoch+1:02d}/{N_EPOCHS}  loss={avg:.4f}")

    return model, losses

print("✅ Helper functions ready")

✅ Helper functions ready


In [ ]:
# ── Main training loop ─────────────────────────────────────
print("=" * 60)
print(f" Phase 4 — Intra-Sector Dense Training")
print(f" Sector : {SECTOR}")
print(f" Years  : 0–19 (2005–2024)")
print(f" Models : {N_YEARS}")
print("=" * 60)

sector_model_dir = os.path.join(MODEL_DIR, SECTOR)
sector_loss_dir  = os.path.join(LOSS_DIR,  SECTOR)
os.makedirs(sector_model_dir, exist_ok=True)
os.makedirs(sector_loss_dir,  exist_ok=True)

total_trained = 0
total_skipped = 0
total_start   = time.time()

for year_idx in range(10,15):
    year_label = 2005 + year_idx
    print(f"\n── Year {year_idx:02d} ({year_label}) ──────────────────────────")

    if is_trained(SECTOR, year_idx):
        print(f"  Already trained ✅ skipping")
        total_skipped += 1
        continue

    year_start    = time.time()
    model, losses = train_one(SECTOR, year_idx)
    elapsed       = time.time() - year_start

    # Save model
    save_path = get_model_path(SECTOR, year_idx)
    torch.save(model.state_dict(), save_path)

    # Save losses
    pd.DataFrame({
        "epoch": range(1, N_EPOCHS + 1),
        "loss" : losses
    }).to_csv(os.path.join(sector_loss_dir,
              f"year_{year_idx:02d}_losses.csv"), index=False)

    total_trained += 1
    print(f"  ✅ Saved  → {save_path}")
    print(f"  Loss     : {losses[0]:.4f} → {losses[-1]:.4f}")
    print(f"  Time     : {elapsed:.1f}s")

 Phase 4 — Intra-Sector Dense Training
 Sector : Healthcare
 Years  : 0–19 (2005–2024)
 Models : 20

── Year 10 (2015) ──────────────────────────
      Epoch 01/10  loss=0.8362
      Epoch 02/10  loss=0.7158
      Epoch 03/10  loss=0.5087
      Epoch 04/10  loss=0.4134
      Epoch 05/10  loss=0.3658
      Epoch 06/10  loss=0.3650
      Epoch 07/10  loss=0.3398
      Epoch 08/10  loss=0.3516
      Epoch 09/10  loss=0.3274
      Epoch 10/10  loss=0.3376
  ✅ Saved  → /content/drive/MyDrive/MSC Thesis/Models_Dense/Healthcare/year_10.pt
  Loss     : 0.8362 → 0.3376
  Time     : 520.7s

── Year 11 (2016) ──────────────────────────
      Epoch 01/10  loss=0.8844
      Epoch 02/10  loss=0.7578
      Epoch 03/10  loss=0.7622
      Epoch 04/10  loss=0.5901
      Epoch 05/10  loss=0.5826
      Epoch 06/10  loss=0.5499
      Epoch 07/10  loss=0.5372
      Epoch 08/10  loss=0.5344
      Epoch 09/10  loss=0.5133
      Epoch 10/10  loss=0.5015
  ✅ Saved  → /content/drive/MyDrive/MSC Thesis/Models_Den


 Summary — Financial Dense Bottleneck W Matrices
 year  tau_dense top_influencer most_influenced
 2005     0.2304             MS               C
 2008     0.4550            PNC             BAC

  τ comparison across years:
  2005 (baseline) : 0.2304
  2008 (GFC)      : 0.4550


IndexError: list index out of range

ValueError: Mountpoint must not already contain files